## Pitch Visualization

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np

# Datenset laden

In [ ]:
data = pd.read_csv('../data/raw/DATASET CSV 11-02-2026.csv', sep=';')
data['br-pos'] = pd.to_numeric(data['br-pos'], errors='coerce')
data['annahme'] = pd.to_numeric(data['annahme'], errors='coerce')

# Überblick über die Daten schaffen

In [ ]:
data.head()
data.shape
data.columns
data.info()
data.isna().sum()
unique_titel = data['titel_kurz_d'].unique()
len(unique_titel)
unique_titel[:20]

# Wrangling und Datacleaning

In [ ]:
data['datum'] = pd.to_datetime(data['datum'], format='%d.%m.%Y')

data.replace([9999,999], np.nan, inplace=True)
# Kongruenz-Spalte erstellen (Regierungstreue)
#br-pos: 10=Ja, 2=Nein | annahme: 1 = Angenommen, 0 = Abgelehnt

data['regierungstreue'] = np.where(
    ((data['br-pos'] == 1) & (data['annahme'] == 1)) |
    ((data['br-pos'] == 2) & (data['annahme'] == 0)),
    1,
    0
)

for col in ['d1e1', 'd1e2', 'd1e3', 'd2e1', 'd2e2', 'd2e3', 'd3e1', 'd3e2', 'd3e3']:
    print(f"\n{col}")
    print(data[col].value_counts(dropna=False).head(20))
    
hauptgruppen_map = {
    1: 'Staatsordnung & Politische Rechte',
    2: 'EU & Aussenpolitik',
    3: 'Armee & Sicherheit',
    4: 'Wirtschaft & Arbeit',
    5: 'Landwirtschaft',
    6: 'Finanzen & Steuern',
    7: 'Energie',
    8: 'Verkehr & Infrastruktur',
    9: 'Klima & Umwelt',
    10: 'Soziale Sicherheit & Gesellschaft',
    11: 'Bildung & Forschung',
    12: 'Kultur, Religion & Medien'
}

data['Hauptgruppe'] = data['d1e1'].map(hauptgruppen_map)


def assign_hauptgruppe(row):
    d1 = row['d1e1']
    d2 = row['d1e2']
    d3 = row['d1e3']

    if d1 == 10:
        if d2 == 1:
            return 'Gesundheit'
        elif d2 == 2:
            return 'Soziale Sicherheit'
        elif d2 == 3:
            return 'Migration & Gesellschaft'
        else:
            return 'Soziale Sicherheit & Gesellschaft'

    elif d1 == 11:
        return 'Bildung & Forschung'
    elif d1 == 12:
        return 'Kultur, Religion & Medien'
    elif d1 == 2:
        return 'EU & Aussenpolitik'
    elif d1 == 3:
        return 'Armee & Sicherheit'
    elif d1 in [7, 9]:
        return 'Klima & Umwelt'
    elif d1 == 6:
        return 'Finanzen & Steuern'
    elif d1 == 8:
        return 'Verkehr & Infrastruktur'
    elif d1 == 4:
        return 'Wirtschaft & Arbeit'
    elif d1 == 5:
        return 'Landwirtschaft'
    elif d1 == 1:
        return 'Staatsordnung & Politische Rechte'
    else:
        return 'Andere'
    
data['Hauptgruppe'] = data.apply(assign_hauptgruppe, axis=1)

data['Hauptgruppe'].value_counts(dropna=False)
data[['titel_kurz_d', 'd1e1', 'd1e2', 'd1e3', 'Hauptgruppe']].sample(20, random_state=42)
data[data['Hauptgruppe'].isna()][['titel_kurz_d', 'd1e1', 'd1e2', 'd1e3']].head(20)

rechtsform_map = {
    1: 'Obligatorisches Referendum',
    2: 'Fakultatives Referendum',
    3: 'Volksinitiative',
    4: 'Direkter Gegenentwurf',
    5: 'Stichfrage'
}

data['rechtsform_name'] = data['rechtsform'].map(rechtsform_map)

kanton_cols = [col for col in data.columns if col.endswith('-japroz')]
print(kanton_cols)
print(len(kanton_cols))

kanton_df = data.melt(
    id_vars=['anr', 'titel_kurz_d', 'regierungstreue', 'br-pos', 'annahme'],
    value_vars=kanton_cols,
    var_name='Kanton',
    value_name='Ja_Anteil'
)

kanton_df['Kanton'] = kanton_df['Kanton'].str.replace('-japroz', '', regex=False).str.upper()


# Vorbereitung für Visualisierungen
plot_data = data[
    data['br-pos'].isin([1, 2]) &
    data['annahme'].isin([0, 1])
].copy()

plot_data['jahr'] = plot_data['datum'].dt.year
plot_data['dekade'] = (plot_data['jahr'] // 10) * 10

plot_data['rechtsform_group'] = plot_data['rechtsform_name'].replace({
    'Obligatorisches Referendum': 'Referendum',
    'Fakultatives Referendum': 'Referendum'
})


# Kantonale Annahme-Spalten finden
kanton_annahme_cols = [col for col in data.columns if col.endswith('-annahme')]

# In Long-Format bringen
kanton_vote_df = data.melt(
    id_vars=['anr', 'titel_kurz_d', 'br-pos'],
    value_vars=kanton_annahme_cols,
    var_name='Kanton',
    value_name='kanton_annahme'
)

# Kantonskürzel bereinigen
kanton_vote_df['Kanton'] = kanton_vote_df['Kanton'].str.replace('-annahme', '', regex=False).str.upper()

# Nur klare Fälle behalten
kanton_vote_df = kanton_vote_df[
    kanton_vote_df['br-pos'].isin([1, 2]) &
    kanton_vote_df['kanton_annahme'].isin([0, 1])
].copy()

# Kantonale Regierungstreue berechnen
kanton_vote_df['kanton_regierungstreue'] = np.where(
    ((kanton_vote_df['br-pos'] == 1) & (kanton_vote_df['kanton_annahme'] == 1)) |
    ((kanton_vote_df['br-pos'] == 2) & (kanton_vote_df['kanton_annahme'] == 0)),
    1,
    0
)

# Sprachregionen zuordnen
sprachregion_map = {
    'ZH': 'Deutschschweiz', 'BE': 'Deutschschweiz', 'LU': 'Deutschschweiz',
    'UR': 'Deutschschweiz', 'SZ': 'Deutschschweiz', 'OW': 'Deutschschweiz',
    'NW': 'Deutschschweiz', 'GL': 'Deutschschweiz', 'ZG': 'Deutschschweiz',
    'FR': 'Romandie', 'SO': 'Deutschschweiz', 'BS': 'Deutschschweiz',
    'BL': 'Deutschschweiz', 'SH': 'Deutschschweiz', 'AR': 'Deutschschweiz',
    'AI': 'Deutschschweiz', 'SG': 'Deutschschweiz', 'GR': 'Deutschschweiz',
    'AG': 'Deutschschweiz', 'TG': 'Deutschschweiz', 'TI': 'Tessin',
    'VD': 'Romandie', 'VS': 'Romandie', 'NE': 'Romandie',
    'GE': 'Romandie', 'JU': 'Romandie'
}

# Pro Kanton mitteln
kanton_summary = (
    kanton_vote_df.groupby('Kanton', as_index=False)['kanton_regierungstreue']
    .mean()
)

kanton_summary['Sprachregion'] = kanton_summary['Kanton'].map(sprachregion_map)

# Nach Regierungstreue sortieren
kanton_summary = kanton_summary.sort_values('kanton_regierungstreue')

# Kantons-Spalten
kanton_cols = [col for col in data.columns if col.endswith('-japroz')]

# Long-Format: jede Abstimmung x jeder Kanton
heatmap_df = data.melt(
    id_vars=['anr', 'titel_kurz_d', 'Hauptgruppe'],
    value_vars=kanton_cols,
    var_name='Kanton',
    value_name='Ja_Anteil'
)

# Kantonsnamen bereinigen
heatmap_df['Kanton'] = heatmap_df['Kanton'].str.replace('-japroz', '', regex=False).str.upper()

# Ja-Anteil numerisch
heatmap_df['Ja_Anteil'] = pd.to_numeric(heatmap_df['Ja_Anteil'], errors='coerce')

# Fehlende Werte entfernen
heatmap_df = heatmap_df.dropna(subset=['Ja_Anteil', 'Hauptgruppe'])

# Durchschnittlicher Ja-Anteil pro Hauptgruppe und Kanton
heatmap_data = (
    heatmap_df.groupby(['Hauptgruppe', 'Kanton'])['Ja_Anteil']
    .mean()
    .reset_index()
    .pivot(index='Hauptgruppe', columns='Kanton', values='Ja_Anteil')
)

# Visualisierungen

### Regierungstreue des Volks im Vergleich zum Bundesrat (pro 10 Jahres Rhytmus) Figure.1

In [ ]:
plot_data = plot_data[plot_data['rechtsform_group'] != 'Direkter Gegenentwurf']

counts = plot_data.groupby('dekade').size()
valid_dekaden = counts[counts >= 5].index

plot_data = plot_data[plot_data['dekade'].isin(valid_dekaden)]

plt.figure(figsize=(12,6))

sns.lineplot(
    data=plot_data,
    x='dekade',
    y='regierungstreue',
    hue='rechtsform_group',
    estimator='mean',
    errorbar=None,
    marker='o',
    linewidth=2.5
)

plt.title('Folgt das Volk dem Bundesrat? Regierungskonformität seit 1870')
plt.legend(title='Abstimmungstyp', loc='upper left')
plt.xlabel('Dekade')
plt.ylabel('Anteil regierungskonformer Abstimmungen')
plt.ylim(0,1)

plt.grid(alpha=0.3)

plt.show()

### Wo vertraut das Volk dem Bundesrat am meisten Figure.2

In [ ]:
plt.figure(figsize=(10,7))

topic_order = (
    plot_data.groupby('Hauptgruppe')['regierungstreue']
    .mean()
    .sort_values()
    .index
)

sns.barplot(
    data=plot_data,
    x='regierungstreue',
    y='Hauptgruppe',
    order=topic_order,
    estimator='mean',
    errorbar=None
)

plt.title('Wo folgt das Volk dem Bundesrat am häufigsten?')
plt.xlabel('Anteil regierungskonformer Abstimmungen')
plt.ylabel('Politikbereich')

plt.xlim(0,1)
plt.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()



### Wo vertraut das Volk dem Bundesrat am meisten Figure.3



In [ ]:
plot_data['abweichung'] = 1 - plot_data['regierungstreue']
plt.figure(figsize=(10,7))

topic_order = (
    plot_data.groupby('Hauptgruppe')['abweichung']
    .mean()
    .sort_values(ascending=False)
    .index
)

sns.barplot(
    data=plot_data,
    x='abweichung',
    y='Hauptgruppe',
    order=topic_order,
    estimator='mean',
    errorbar=None
)

plt.title('Wo widerspricht das Volk dem Bundesrat am häufigsten?')
plt.xlabel('Anteil nicht regierungskonformer Abstimmungen')
plt.ylabel('Politikbereich')
plt.xlim(0,1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()



### Wo vertraut das Volk dem Bundesrat am meisten Figure.4

In [ ]:
topic_summary = (
    plot_data.groupby('Hauptgruppe')['regierungstreue']
    .mean()
    .reset_index()
)

topic_summary['abweichung'] = 1 - topic_summary['regierungstreue']
topic_summary = topic_summary.sort_values('regierungstreue')

topic_long = topic_summary.melt(
    id_vars='Hauptgruppe',
    value_vars=['regierungstreue', 'abweichung'],
    var_name='Typ',
    value_name='Anteil'
)

plt.figure(figsize=(11,7))

sns.barplot(
    data=topic_long,
    x='Anteil',
    y='Hauptgruppe',
    hue='Typ'
)

plt.title('Folgt das Volk dem Bundesrat oder widerspricht es ihm?')
plt.xlabel('Anteil der Abstimmungen')
plt.ylabel('Politikbereich')
plt.xlim(0,1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()




### Gibt es regionale Unterschiede darin, wie stark Kantone der Empfehlung des Bundesrats folgen? (Boxplot) 
ungeeignet habe ihn dennoch mal drin gelassen

In [ ]:
plt.figure(figsize=(14, 7))

sns.boxplot(
    data=kanton_df,
    x='Kanton',
    y='Ja_Anteil'
)

plt.title('Verteilung der Ja-Anteile nach Kanton')
plt.xlabel('Kanton')
plt.ylabel('Ja-Stimmen-Anteil')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()



### Wie häufig folgen Kantone dem Bundesrat?


In [ ]:
# FacetGrid nach Sprachregion
g = sns.FacetGrid(
    kanton_summary,
    col="Sprachregion",
    col_wrap=3,
    height=5,
    sharex=True,
    sharey=False
)

g.map_dataframe(
    sns.barplot,
    x="kanton_regierungstreue",
    y="Kanton",
    order=kanton_summary.sort_values('kanton_regierungstreue')['Kanton']
)

g.set_axis_labels("Anteil regierungskonformer Abstimmungen", "Kanton")
g.set_titles("{col_name}")

g.fig.suptitle(
    "Kantonale Regierungstreue nach Sprachregion",
    y=1.03,
    fontsize=14
)

plt.xlim(0,1)
plt.tight_layout()
plt.show()


# Plot
plt.figure(figsize=(14, 8))

sns.barplot(
    data=kanton_summary,
    x='kanton_regierungstreue',
    y='Kanton',
    hue='Sprachregion'
)

plt.title('Kantonale Regierungstreue: Wie häufig folgen Kantone dem Bundesrat?')
plt.xlabel('Anteil kantonal regierungskonformer Abstimmungen')
plt.ylabel('Kanton')
plt.xlim(0, 1)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Visualisierung Nr. 3 – Der Röstigraben des Vertrauens

In [ ]:
# FacetGrid nach Sprachregion
g = sns.FacetGrid(
    kanton_summary,
    col="Sprachregion",
    col_wrap=3,
    height=5,
    sharex=True,
    sharey=False
)

g.map_dataframe(
    sns.barplot,
    x="kanton_regierungstreue",
    y="Kanton",
    order=kanton_summary.sort_values('kanton_regierungstreue')['Kanton']
)

g.set_axis_labels("Anteil regierungskonformer Abstimmungen", "Kanton")
g.set_titles("{col_name}")

g.fig.suptitle(
    "Der Röstigraben des Vertrauens: Kantonale Regierungstreue nach Sprachregion",
    y=1.03,
    fontsize=14
)

for ax in g.axes.flat:
    ax.set_xlim(0, 1)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()




### Gibt es Regionale Unterschiede darin,wie stark die Kantone der Empfehlung des Bundesrates folgen?? Figure.5

In [ ]:
plt.figure(figsize=(16, 8))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn",
    center=50
)

plt.title("Durchschnittlicher Ja-Stimmen-Anteil nach Hauptgruppe und Kanton")
plt.xlabel("Kanton")
plt.ylabel("Hauptgruppe")
plt.tight_layout()
plt.show()